## Activity 2: Design, Query, and Validate a Relational Database

### Dataset Load

In [4]:
import pandas as pd
import sqlite3

# Load the original Module 2 dataset
df = pd.read_csv(
    "../data/raw/seattle_building_energy_original.csv",
    low_memory=False
)

# Check if the dataset loaded successfully
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (38309, 46)


,OSEBuildingID,DataYear,BuildingName,BuildingType,TaxParcelIdentificationNumber,Address,City,State,ZipCode,Latitude,...,Electricity(kWh),SteamUse(kBtu),NaturalGas(therms),ComplianceStatus,ComplianceIssue,Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,GHGEmissionsIntensity,Demolished
0,1,2025,Mayflower Park Hotel,NonResidential,659000030,405 OLIVE WAY,SEATTLE,WA,98101,47.612199,...,"1,021,340",NaN,"16,927.4",Not Compliant,NaN,"3,484,810.7","1,692,740",NaN,NaN,False
1,2,2025,Paramount Hotel,NonResidential,659000220,724 PINE ST,SEATTLE,WA,98101,47.613073,...,"775,602",NaN,"34,847",Compliant,No Issue,"2,646,356","3,484,730",199.75,2.26,False
2,3,2025,Westin Hotel (Parent Building),NonResidential,659000475,1900 5TH AVE,SEATTLE,WA,98101,47.613672,...,"11,693,500","29,336,926","51,470",Compliant,No Issue,"39,898,222","5,147,040","2,880.5",3.48,False
3,5,2025,Hotel Max,NonResidential,659000640,620 STEWART ST,SEATTLE,WA,98101,47.614121,...,"772,199","1,349,793","11,410",Compliant,No Issue,"2,634,742","1,141,040",184.94,3.02,False
4,8,2025,Warwick Seattle Hotel,NonResidential,659000970,401 LENORA ST,SEATTLE,WA,98121,47.613950,...,"1,394,922",NaN,"58,654",Compliant,No Issue,"4,759,474","5,865,420",338.2,2.74,False


### Task 1 – Apply the Four Design Decisions to Your Data

#### 1.1 Identify the business process

1. Business Process

The business process represented by my dataset is building energy benchmarking and reporting. It records the energy performance and characteristics of buildings in Seattle for different reporting years.

2. Grain

One row represents one building-year benchmarking record. A building may have multiple records for different reporting years.

3. Dimensions

The dimensions provide descriptive information that helps me analyze and group the records.

- Building
- Reporting year
- Building type
- Property use
- Building location

4. Facts

The facts are the numerical measurements used for analysis.

- Site Energy Use Intensity (Site EUI)
- Source Energy Use Intensity (Source EUI)
- Site Energy Use
- Total GHG Emissions
- Electricity consumption
- Natural gas consumption

These facts allow me to compare energy performance across building types and reporting years.

In [5]:
import pandas as pd
import sqlite3

task1_design = pd.DataFrame({
    "Design Decision": [
        "1. Business Process",
        "2. Grain",
        "3. Dimensions",
        "4. Facts"
    ],
    "Seattle Building Energy Dataset": [
        "Building energy benchmarking and reporting",
        "One building-year benchmarking record",
        "Building, reporting year, building type, property use, location",
        "Site EUI, Source EUI, Site Energy Use, GHG Emissions, energy consumption"
    ]
})

display(task1_design)

,Design Decision,Seattle Building Energy Dataset
0,1. Business Process,Building energy benchmarking and reporting
1,2. Grain,One building-year benchmarking record
2,3. Dimensions,"Building, reporting year, building type, prope..."
3,4. Facts,"Site EUI, Source EUI, Site Energy Use, GHG Emi..."


### Task 2 — Choose tables and keys

#### 2.1 Verify the Building Primary Key

 Table Structure and Relationships

| Table | One Row Represents | Description |
|---|---|---|
| `Building` | One unique building | Stores information about each building. |
| `Benchmark` | One building's benchmarking record for one reporting year | Stores the energy benchmarking data of a building for a specific year. |

 Table Relationship
One building can have many benchmarking records across different reporting years (one-to-many relationship).

#### 2.2 Verify the Building Primary Key

I propose using OSEBuildingID as the primary key of the Building table because it identifies a building in the dataset.

To verify this, I will check whether OSEBuildingID contains unique values and whether any building identifiers are missing.

If every building identifier is unique in the building-level dataset, OSEBuildingID can serve as its primary key.

In [7]:
# Check whether OSEBuildingID identifies unique buildings

building_id_check = pd.DataFrame({
    "Check": [
        "Total records",
        "Unique building IDs",
        "Missing building IDs",
        "Duplicate building IDs"
    ],
    "Result": [
        len(df),
        df["OSEBuildingID"].nunique(),
        df["OSEBuildingID"].isna().sum(),
        df["OSEBuildingID"].duplicated().sum()
    ]
})

display(building_id_check)

,Check,Result
0,Total records,38309
1,Unique building IDs,3871
2,Missing building IDs,0
3,Duplicate building IDs,34438


#### 2.3 Verify the Benchmark Primary Key

I propose using OSEBuildingID and DataYear together as the composite primary key of the Benchmark table.

OSEBuildingID alone cannot identify a benchmarking record because the same building may appear in multiple reporting years. Combining OSEBuildingID with DataYear should identify one benchmarking record for a particular building and year.

I will verify this by checking for duplicate building-year combinations.

In [8]:
# Check the composite key

composite_key_check = pd.DataFrame({
    "Check": [
        "Total building-year records",
        "Unique building-year combinations",
        "Duplicate building-year records"
    ],
    "Result": [
        len(df),
        df[["OSEBuildingID", "DataYear"]].drop_duplicates().shape[0],
        df.duplicated(subset=["OSEBuildingID", "DataYear"]).sum()
    ]
})

display(composite_key_check)

,Check,Result
0,Total building-year records,38309
1,Unique building-year combinations,38309
2,Duplicate building-year records,0


#### 2.4 Functional Dependencies

I expect OSEBuildingID to identify building-related information and the combination of OSEBuildingID and DataYear to identify annual benchmarking information.

To support the proposed Building table, I will examine whether selected building attributes have consistent values for each building across reporting years.

If a building attribute changes over time, I will document this rather than automatically assume that the attribute is permanently determined by OSEBuildingID.

In [9]:
# Check whether selected building attributes are consistent per building

building_attributes = [
    "BuildingName",
    "BuildingType",
    "Address",
    "PropertyGFATotal"
]

dependency_check = (
    df.groupby("OSEBuildingID")[building_attributes]
    .nunique(dropna=True)
    .reset_index()
)

dependency_summary = pd.DataFrame({
    "Attribute": building_attributes,
    "Buildings with Multiple Values": [
        (dependency_check[col] > 1).sum()
        for col in building_attributes
    ]
})

display(dependency_summary)

,Attribute,Buildings with Multiple Values
0,BuildingName,3410
1,BuildingType,0
2,Address,54
3,PropertyGFATotal,1


#### 2.5 Proposed Tables and Keys

I propose separating the dataset into two relational tables.

The Building table will contain one record per building, identified by OSEBuildingID. The Benchmark table will contain one record per building-year, identified by the composite key OSEBuildingID and DataYear.

The Benchmark table will reference OSEBuildingID from the Building table as a foreign key. This creates a one-to-many relationship between Building and Benchmark.

The design reduces repeated building information and allows annual energy performance to be analyzed separately from building-level characteristics.

#### Database Tables and Keys

| Table | One Row = | Primary Key | Foreign Key(s) | Evidence the Key Is Unique |
|---|---|---|---|---|
| `Building` | One unique building | `OSEBuildingID` | None | One record per OSEBuildingID after removing redundant records. |
| `Benchmark` | One building-year benchmarking record | `OSEBuildingID` + `DataYear` | `OSEBuildingID` → `Building.OSEBuildingID` | 38,309 unique building-year combinations; 0 duplicate combinations. |

### Task 3 — Design the schema

In [15]:
import sqlite3
import pandas as pd

# Create or connect to the database
con = sqlite3.connect('../db/module3.db')

# Enable foreign key enforcement
con.execute("PRAGMA foreign_keys = ON")

# Create the tables
con.executescript("""
CREATE TABLE IF NOT EXISTS Building (
    building_id INTEGER PRIMARY KEY,
    building_name TEXT,
    building_type TEXT,
    address TEXT,
    property_gfa_total REAL
);

CREATE TABLE IF NOT EXISTS Benchmark (
    building_id INTEGER NOT NULL,
    data_year INTEGER NOT NULL,
    site_eui REAL,
    source_eui REAL,
    site_energy_use REAL,
    total_ghg_emissions REAL,

    PRIMARY KEY (building_id, data_year),

    FOREIGN KEY (building_id)
        REFERENCES Building(building_id)
);
""")

print("Database and tables created successfully.")

Database and tables created successfully.


In [17]:
# Check database tables
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table';
""", con)

building_schema = pd.read_sql_query(
    "PRAGMA table_info(Building);",
    con
)

benchmark_schema = pd.read_sql_query(
    "PRAGMA table_info(Benchmark);",
    con
)

foreign_key_check = pd.read_sql_query(
    "PRAGMA foreign_key_list(Benchmark);",
    con
)
display(tables)
display(building_schema)
display(benchmark_schema)
display(foreign_key_check)


,name
0,Building
1,Benchmark


,cid,name,type,notnull,dflt_value,pk
0,0,building_id,INTEGER,0,None,1
1,1,building_name,TEXT,0,None,0
2,2,building_type,TEXT,0,None,0
3,3,address,TEXT,0,None,0
4,4,property_gfa_total,REAL,0,None,0


,cid,name,type,notnull,dflt_value,pk
0,0,building_id,INTEGER,1,None,1
1,1,data_year,INTEGER,1,None,2
2,2,site_eui,REAL,0,None,0
3,3,source_eui,REAL,0,None,0
4,4,site_energy_use,REAL,0,None,0
5,5,total_ghg_emissions,REAL,0,None,0


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,Building,building_id,building_id,NO ACTION,NO ACTION,NONE


### Task 4 — Load and reconcile

##### 4.1 Data Preparation and Type Conversion

In [56]:
import pandas as pd
import sqlite3
from pathlib import Path

# Database path
db_path = Path("../db/module3.db")

# Connect to the database
con = sqlite3.connect(db_path)

# Enable foreign-key enforcement
con.execute("PRAGMA foreign_keys = ON")

print("Database connection established.")


Database connection established.


In [57]:

# Check source dataset dimensions
print("Source dataset shape:", df.shape)

# Check the source columns needed for Task 4
required_columns = [
    "OSEBuildingID",
    "DataYear",
    "BuildingName",
    "BuildingType",
    "Address",
    "PropertyGFATotal",
    "SiteEUI(kBtu/sf)",
    "SourceEUI(kBtu/sf)",
    "SiteEnergyUse(kBtu)",
    "TotalGHGEmissions"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing required columns:", missing_columns)


Source dataset shape: (38309, 46)
Missing required columns: []


In [58]:

# Create a working copy
df_prepared = df[required_columns].copy()

# Convert building ID and reporting year
df_prepared["OSEBuildingID"] = pd.to_numeric(
    df_prepared["OSEBuildingID"],
    errors="coerce"
).astype("Int64")

df_prepared["DataYear"] = pd.to_numeric(
    df_prepared["DataYear"],
    errors="coerce"
).astype("Int64")

# Convert numeric benchmarking measurements
measurement_columns = [
    "SiteEUI(kBtu/sf)",
    "SourceEUI(kBtu/sf)",
    "SiteEnergyUse(kBtu)",
    "TotalGHGEmissions"
]

for col in measurement_columns:
    df_prepared[col] = pd.to_numeric(
        df_prepared[col],
        errors="coerce"
    )

# Convert total property floor area
df_prepared["PropertyGFATotal"] = pd.to_numeric(
    df_prepared["PropertyGFATotal"]
    .astype("string")
    .str.replace(",", "", regex=False),
    errors="coerce"
)

print("Data preparation completed.")

display(df_prepared.head())


Data preparation completed.


,OSEBuildingID,DataYear,BuildingName,BuildingType,Address,PropertyGFATotal,SiteEUI(kBtu/sf),SourceEUI(kBtu/sf),SiteEnergyUse(kBtu),TotalGHGEmissions
0,1,2025,Mayflower Park Hotel,NonResidential,405 OLIVE WAY,88434,NaN,NaN,NaN,NaN
1,2,2025,Paramount Hotel,NonResidential,724 PINE ST,103566,69.3,125.1,NaN,199.75
2,3,2025,Westin Hotel (Parent Building),NonResidential,1900 5TH AVE,956110,89.9,184.1,NaN,NaN
3,5,2025,Hotel Max,NonResidential,620 STEWART ST,61320,83.6,166.3,NaN,184.94
4,8,2025,Warwick Seattle Hotel,NonResidential,401 LENORA ST,175580,86.1,157.8,NaN,338.20


##### 4.2 Building Attribute Consistency Check

Before creating the Building table, I examined whether building-related attributes remain consistent for the same OSEBuildingID.

The attributes checked were BuildingName, BuildingType, Address, and PropertyGFATotal. The number of distinct non-missing values for each attribute was calculated per building.

This check helps identify attributes that may change or vary across building-year records. For the Building table, the first non-missing value for each attribute is retained for each building ID.

In [59]:

building_attributes = [
    "BuildingName",
    "BuildingType",
    "Address",
    "PropertyGFATotal"
]

# Count distinct non-missing values per building
attribute_check = (
    df_prepared.groupby("OSEBuildingID")[building_attributes]
    .nunique(dropna=True)
)

# Summarize the number of buildings with multiple values
attribute_summary = pd.DataFrame({
    "Attribute": building_attributes,
    "Buildings with Multiple Values": [
        (attribute_check[col] > 1).sum()
        for col in building_attributes
    ]
})

display(attribute_summary)


,Attribute,Buildings with Multiple Values
0,BuildingName,3410
1,BuildingType,0
2,Address,54
3,PropertyGFATotal,1


##### 4.3 Prepare Building and Benchmark Tables

In [60]:

building_columns = [
    "OSEBuildingID",
    "BuildingName",
    "BuildingType",
    "Address",
    "PropertyGFATotal"
]

# Retain the first non-missing value per building attribute
building_df = (
    df_prepared[building_columns]
    .groupby("OSEBuildingID", as_index=False)
    .first()
)

# Rename columns to match the database schema
building_df = building_df.rename(columns={
    "OSEBuildingID": "building_id",
    "BuildingName": "building_name",
    "BuildingType": "building_type",
    "Address": "address",
    "PropertyGFATotal": "property_gfa_total"
})

print("Prepared Building records:", len(building_df))

display(building_df.head())

Prepared Building records: 3871


,building_id,building_name,building_type,address,property_gfa_total
0,1,Mayflower Park Hotel,NonResidential,405 OLIVE WAY,88434
1,2,Paramount Hotel,NonResidential,724 PINE ST,103566
2,3,Westin Hotel (Parent Building),NonResidential,1900 5TH AVE,956110
3,5,Hotel Max,NonResidential,620 STEWART ST,61320
4,8,Warwick Seattle Hotel,NonResidential,401 LENORA ST,175580


In [61]:
benchmark_columns = [
    "OSEBuildingID",
    "DataYear",
    "SiteEUI(kBtu/sf)",
    "SourceEUI(kBtu/sf)",
    "SiteEnergyUse(kBtu)",
    "TotalGHGEmissions"
]

# Retain all source building-year records
benchmark_df = df_prepared[benchmark_columns].copy()

# Rename columns to match the database schema
benchmark_df = benchmark_df.rename(columns={
    "OSEBuildingID": "building_id",
    "DataYear": "data_year",
    "SiteEUI(kBtu/sf)": "site_eui",
    "SourceEUI(kBtu/sf)": "source_eui",
    "SiteEnergyUse(kBtu)": "site_energy_use",
    "TotalGHGEmissions": "total_ghg_emissions"
})

print("Prepared Benchmark records:", len(benchmark_df))

display(benchmark_df.head())

Prepared Benchmark records: 38309


,building_id,data_year,site_eui,source_eui,site_energy_use,total_ghg_emissions
0,1,2025,NaN,NaN,NaN,NaN
1,2,2025,69.3,125.1,NaN,199.75
2,3,2025,89.9,184.1,NaN,NaN
3,5,2025,83.6,166.3,NaN,184.94
4,8,2025,86.1,157.8,NaN,338.20


##### 4.4 Check Prepared Data Before Loading

Before inserting the prepared data into SQLite, I checked whether the proposed primary keys contain duplicate or missing values.

The Building table must have unique and non-missing building IDs. The Benchmark table must have unique building-year combinations and no missing values in either component of its composite primary key.

In [62]:

preload_check = pd.DataFrame({
    "Table": ["Building", "Benchmark"],
    "Prepared Records": [
        len(building_df),
        len(benchmark_df)
    ],
    "Duplicate Keys": [
        building_df["building_id"].duplicated().sum(),
        benchmark_df.duplicated(
            subset=["building_id", "data_year"]
        ).sum()
    ],
    "Missing Primary Key Values": [
        building_df["building_id"].isna().sum(),
        (
            benchmark_df["building_id"].isna()
            | benchmark_df["data_year"].isna()
        ).sum()
    ]
})

display(preload_check)

,Table,Prepared Records,Duplicate Keys,Missing Primary Key Values
0,Building,3871,0,0
1,Benchmark,38309,0,0


##### 4.5 Load Data into SQLite

In [65]:
# Clear Benchmark first because it references Building
con.execute("DELETE FROM Benchmark")

# Clear Building after Benchmark
con.execute("DELETE FROM Building")

# Save the changes
con.commit()

print("Existing records cleared.")

Existing records cleared.


In [66]:
building_df.to_sql(
    "Building",
    con,
    if_exists="append",
    index=False
)

# Load Benchmark records
benchmark_df.to_sql(
    "Benchmark",
    con,
    if_exists="append",
    index=False
)

# Commit changes
con.commit()

print("Building and Benchmark records loaded successfully.")


Building and Benchmark records loaded successfully.


##### 4.6 Validate Database Loading

In [67]:

# Source record counts
source_building_count = df_prepared["OSEBuildingID"].nunique()
source_benchmark_count = len(df_prepared)

# Loaded database record counts
loaded_building_count = pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM Building",
    con
)["count"].iloc[0]

loaded_benchmark_count = pd.read_sql_query(
    "SELECT COUNT(*) AS count FROM Benchmark",
    con
)["count"].iloc[0]

# Compare source and loaded records
row_count_check = pd.DataFrame({
    "Table": ["Building", "Benchmark"],
    "Source Records": [
        source_building_count,
        source_benchmark_count
    ],
    "Loaded Records": [
        loaded_building_count,
        loaded_benchmark_count
    ]
})

row_count_check["Difference"] = (
    row_count_check["Source Records"]
    - row_count_check["Loaded Records"]
)

display(row_count_check)


,Table,Source Records,Loaded Records,Difference
0,Building,3871,3871,0
1,Benchmark,38309,38309,0


In [68]:

# Check database integrity
integrity_check = pd.read_sql_query(
    "PRAGMA integrity_check;",
    con
)

# Check foreign-key violations
foreign_key_check = pd.read_sql_query(
    "PRAGMA foreign_key_check;",
    con
)

print("Database Integrity Check:")
display(integrity_check)

print("Foreign-Key Violations:")
display(foreign_key_check)


Database Integrity Check:


,integrity_check
0,ok


Foreign-Key Violations:


,table,rowid,parent,fkid


In [69]:

benchmark_count = pd.read_sql_query("""
SELECT COUNT(*) AS benchmark_records
FROM Benchmark;
""", con)

display(benchmark_count)

,benchmark_records
0,38309


In [70]:

con.close()

print("Database connection closed.")

Database connection closed.


##### 4.7 Task 4 Findings Summary

Table Summary

| Table | Source rows | Loaded rows | Type conversions / notes |
| :--- | :--- | :--- | :--- |
| **Building** | 3,871 | 3,871 | `OSEBuildingID` converted to integer and renamed `building_id`. `PropertyGFATotal` converted to numeric (float). Building attributes extracted using the first non-missing value per building. |
| **Benchmark** | 38,309 | 38,309 | `OSEBuildingID` converted to integer; `DataYear` converted to integer; `Site EUI`, `Source EUI`, `Site Energy Use`, and `Total GHG Emissions` converted to numeric (float). |
| **Total** | **42,180** | **42,180** | **Both tables loaded successfully with matching source and loaded row counts.** |


The Seattle building energy benchmarking dataset was successfully prepared and loaded into the SQLite database. The original dataset contained 38,309 building-year records representing 3,871 unique buildings.

During data preparation, the source columns were converted into appropriate numeric data types, and the original dataset was separated into Building and Benchmark tables. The Building table contains one record per building, while the Benchmark table preserves the building-year records.

The building attribute consistency check identified variations in BuildingName, Address, and PropertyGFATotal across records associated with the same building ID. To maintain one record per building, the first non-missing value for each building attribute was retained. BuildingType showed no multiple-value cases.

Before loading, both prepared tables were checked for duplicate and missing primary key values. No duplicate keys or missing primary key values were detected.

After loading, the record count reconciliation confirmed that all 3,871 Building records and 38,309 Benchmark records were successfully inserted. The source and loaded record counts matched, resulting in zero differences for both tables.

The database integrity check returned `ok`, and the foreign-key validation reported no violations. The Building table also contained 3,871 non-missing property floor area values.

Overall, the results indicate that the data loading and reconciliation process was successful. The relational database is now prepared for SQL querying, joining, aggregation, and further data analysis.

### Task 5 — Write and explain six SQL queries

##### 5.1 Introduction to SQL Querying

SQL (Structured Query Language) is used to retrieve, filter, sort, calculate, and summarize data stored in relational databases.

In this task, SQL queries will be used to explore the Seattle Building Energy Benchmarking dataset and answer questions about building energy performance.

The queries will use the Building and Benchmark tables created in Task 3 and populated in Task 4.

##### 5.2 Query A – Select Specific Columns

Question: What are the building IDs, reporting years, and Site EUI values of the benchmarked buildings?

In [50]:
con = sqlite3.connect("../db/module3.db")

query_a = """
SELECT
    building_id,
    data_year,
    site_eui
FROM Benchmark;
"""

result_a = pd.read_sql_query(query_a, con)

display(result_a.head(10))

,building_id,data_year,site_eui
0,1,2025,NaN
1,2,2025,69.3
2,3,2025,89.9
3,5,2025,83.6
4,8,2025,86.1
5,9,2025,172.1
6,11,2025,71.3
7,12,2025,72.7
8,13,2025,78.6
9,15,2025,60.4


##### 5.3 Query B – Filter Records Using WHERE

Question: Which building-year records have a Site EUI greater than 100 kBtu/sf?

In [51]:
query_b = """
SELECT
    building_id,
    data_year,
    site_eui
FROM Benchmark
WHERE site_eui > 100;
"""

result_b = pd.read_sql_query(query_b, con)

display(result_b.head(10))
print("Records with Site EUI > 100:", len(result_b))

,building_id,data_year,site_eui
0,9,2025,172.1
1,19,2025,113.9
2,24,2025,122.7
3,27,2025,127.9
4,60,2025,162.2
5,84,2025,149.0
6,89,2025,101.5
7,95,2025,305.1
8,96,2025,141.4
9,131,2025,101.5


Records with Site EUI > 100: 2953


##### 5.4 Query C – Sort Records Using ORDER BY

Question: Which 10 building-year records have the highest Site EUI?

In [52]:
query_c = """
SELECT
    building_id,
    data_year,
    site_eui
FROM Benchmark
WHERE site_eui IS NOT NULL
ORDER BY site_eui DESC
LIMIT 10;
"""

result_c = pd.read_sql_query(query_c, con)

display(result_c)

,building_id,data_year,site_eui
0,365,2023,989.200012
1,365,2024,969.799988
2,365,2018,958.500000
3,365,2022,956.700012
4,365,2019,925.099976
5,365,2017,924.400024
6,365,2020,910.900024
7,365,2021,909.900024
8,365,2016,834.400024
9,49780,2017,833.700012


##### 5.5 Query D – Calculate a Derived Value

Question: What is the calculated energy use per square foot for each building-year record with available energy use and floor area?

In [53]:
query_d = """
SELECT
    B.building_id,
    B.data_year,
    B.site_energy_use,
    D.property_gfa_total,
    B.site_energy_use / D.property_gfa_total
        AS calculated_energy_per_sqft
FROM Benchmark B
JOIN Building D
    ON B.building_id = D.building_id
WHERE B.site_energy_use IS NOT NULL
  AND D.property_gfa_total > 0;
"""

result_d = pd.read_sql_query(query_d, con)

display(result_d.head(10))

,building_id,data_year,site_energy_use,property_gfa_total,calculated_energy_per_sqft
0,23458,2020,551.0,22113.0,0.024917
1,21578,2015,433.0,41221.0,0.010504


##### 5.6 Query E – Group and Aggregate Using GROUP BY

Question: What is the average Site EUI and number of benchmark records for each building type?

In [54]:
query_e = """
SELECT
    D.building_type,
    COUNT(B.site_eui) AS records_with_site_eui,
    ROUND(AVG(B.site_eui), 2) AS average_site_eui
FROM Building D
JOIN Benchmark B
    ON D.building_id = B.building_id
GROUP BY D.building_type
ORDER BY average_site_eui DESC;
"""

result_e = pd.read_sql_query(query_e, con)

display(result_e)

,building_type,records_with_site_eui,average_site_eui
0,Hospital,554,147.89
1,University,314,108.18
2,NonResidential,14313,66.70
3,Multifamily HR (10+),1377,48.77
4,School,1463,41.72
5,Multifamily MR (5-9),7531,38.06
6,Multifamily LR (1-4),11168,35.77


##### 5.7 Query F – Join Related Tables

Question: What are the building names, building types, reporting years, and Site EUI values of the benchmarked buildings?

In [55]:
query_f = """
SELECT
    D.building_name,
    D.building_type,
    B.data_year,
    B.site_eui
FROM Building D
INNER JOIN Benchmark B
    ON D.building_id = B.building_id
LIMIT 10;
"""

result_f = pd.read_sql_query(query_f, con)

display(result_f)

,building_name,building_type,data_year,site_eui
0,Mayflower Park Hotel,NonResidential,2025,NaN
1,PARAMOUNT HOTEL,NonResidential,2025,69.3
2,WESTIN HOTEL (Parent Building),NonResidential,2025,89.9
3,HOTEL MAX,NonResidential,2025,83.6
4,WARWICK SEATTLE HOTEL,NonResidential,2025,86.1
5,WEST PRECINCT (SEATTLE POLICE),NonResidential,2025,172.1
6,PARAMOUNT THEATRE,NonResidential,2025,71.3
7,COURTYARD BY MARRIOTT - PIONEER SQ,NonResidential,2025,72.7
8,LYON BUILDING,Multifamily MR (5-9),2025,78.6
9,HOTEL MONACO,NonResidential,2025,60.4


##### 5.8 Task 5 Findings Summary

| Query | SQL Feature | Question Answered | Key Finding |
|---|---|---|---|
| A | SELECT | What are the building IDs, reporting years, and Site EUI values? | Retrieved selected columns from the Benchmark table, including records with missing Site EUI. |
| B | WHERE | Which records have Site EUI greater than 100 kBtu/sf? | Identified building-year observations with relatively high Site EUI. |
| C | ORDER BY | Which 10 building-year records have the highest Site EUI? | Building ID 365 had the highest displayed Site EUI, approximately 989.2 kBtu/sf in 2023. |
| D | Derived Value | What is the calculated energy use per square foot? | Successfully calculated energy use per square foot for two building-year records with available Site Energy Use and positive floor area. |
| E | GROUP BY | What is the average Site EUI by building type? | Hospitals had the highest average Site EUI (147.89), followed by universities (108.18). Low-rise multifamily buildings had the lowest (35.77). |
| F | JOIN | What are the building names, types, reporting years, and Site EUI values? | Successfully combined Building and Benchmark tables using building_id. |

Overall, Task 5 demonstrated how SQL can retrieve, filter, sort, calculate, summarize, and integrate data from relational tables. The queries also revealed differences in energy performance across building types and highlighted a data preparation issue involving property floor area.

### Task 6 — Investigate the join

##### 6.1 Introduction to Joining Relational Tables

A JOIN combines records from two or more relational tables using a related column. In this database, the Building and Benchmark tables are related through building_id.

The Building table contains building information, while the Benchmark table contains annual energy benchmarking measurements. Joining these tables allows building characteristics to be analyzed together with their annual energy performance.

The purpose of this task is to investigate the relationship between the two tables, verify the join results, and determine whether the join preserves the expected records.

##### 6.2 Identify the Relationship Between the Tables

The relationship between the **Building** and **Benchmark** tables is **one-to-many**.
- One building can have multiple benchmarking records across reporting years.
- Each benchmarking record belongs to one building.
- `building_id` is the primary key of the **Building** table.
- `building_id` is a foreign key in the **Benchmark** table.
- The composite primary key of **Benchmark** is `(building_id, data_year)`.

| Table | Key | Relationship |
| :--- | :--- | :--- |
| **Building** | `building_id` (PK) | One building |
| **Benchmark** | `building_id`, `data_year` (PK) | Multiple yearly records per building |
| **Relationship** | `Building.building_id = Benchmark.building_id` | One-to-many |


##### 6.3 Verify the Relationship Using SQL

In [71]:
import sqlite3
import pandas as pd

# Reopen the database if the Task 4 connection was closed
con = sqlite3.connect("../db/module3.db")

# Enable foreign-key enforcement for this connection
con.execute("PRAGMA foreign_keys = ON")

# Check the number of records in each table
table_counts = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM Building) AS building_records,
    (SELECT COUNT(*) FROM Benchmark) AS benchmark_records;
""", con)

display(table_counts)


,building_records,benchmark_records
0,3871,38309


##### 6.4 Perform an INNER JOIN

In [73]:
query_inner_join = """
SELECT
    D.building_id,
    D.building_name,
    D.building_type,
    D.property_gfa_total,
    B.data_year,
    B.site_eui,
    B.source_eui,
    B.site_energy_use,
    B.total_ghg_emissions
FROM Building AS D
INNER JOIN Benchmark AS B
    ON D.building_id = B.building_id
ORDER BY D.building_id, B.data_year;
"""

inner_join_result = pd.read_sql_query(
    query_inner_join,
    con
)

print("INNER JOIN records:", len(inner_join_result))

display(inner_join_result.head(10))


INNER JOIN records: 38309


,building_id,building_name,building_type,property_gfa_total,data_year,site_eui,source_eui,site_energy_use,total_ghg_emissions
0,1,Mayflower Park Hotel,NonResidential,88434.0,2015,78.900002,173.500000,NaN,252.7
1,1,Mayflower Park Hotel,NonResidential,88434.0,2016,81.699997,182.500000,NaN,252.9
2,1,Mayflower Park Hotel,NonResidential,88434.0,2017,83.199997,169.300003,NaN,263.2
3,1,Mayflower Park Hotel,NonResidential,88434.0,2018,83.800003,168.899994,NaN,266.7
4,1,Mayflower Park Hotel,NonResidential,88434.0,2019,83.099998,167.600006,NaN,267.3
5,1,Mayflower Park Hotel,NonResidential,88434.0,2020,54.599998,113.900002,NaN,167.5
6,1,Mayflower Park Hotel,NonResidential,88434.0,2021,73.599998,144.199997,NaN,243.2
7,1,Mayflower Park Hotel,NonResidential,88434.0,2022,82.900002,165.600006,NaN,271.7
8,1,Mayflower Park Hotel,NonResidential,88434.0,2023,90.099998,170.199997,NaN,332.1
9,1,Mayflower Park Hotel,NonResidential,88434.0,2024,61.700001,121.400002,NaN,263.3


##### 6.5 Validate the INNER JOIN Record Count

In [74]:
# Count Benchmark records
benchmark_records = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM Benchmark;
""", con)["total"].iloc[0]

# Count INNER JOIN records
inner_join_records = len(inner_join_result)

# Reconcile the counts
join_count_check = pd.DataFrame({
    "Record Type": [
        "Benchmark records",
        "INNER JOIN records"
    ],
    "Count": [
        benchmark_records,
        inner_join_records
    ]
})

display(join_count_check)

print(
    "Record count difference:",
    benchmark_records - inner_join_records
)


,Record Type,Count
0,Benchmark records,38309
1,INNER JOIN records,38309


Record count difference: 0


##### 6.6 Check for Benchmark Records Without a Matching Building

In [75]:

query_unmatched_benchmark = """
SELECT
    B.building_id,
    B.data_year,
    B.site_eui
FROM Benchmark AS B
LEFT JOIN Building AS D
    ON B.building_id = D.building_id
WHERE D.building_id IS NULL;
"""

unmatched_benchmark = pd.read_sql_query(
    query_unmatched_benchmark,
    con
)

print(
    "Benchmark records without a matching Building:",
    len(unmatched_benchmark)
)

display(unmatched_benchmark.head())


Benchmark records without a matching Building: 0


,building_id,data_year,site_eui


##### 6.7 Compare INNER JOIN and LEFT JOIN

In [76]:

# INNER JOIN count
inner_count = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM Building AS D
INNER JOIN Benchmark AS B
    ON D.building_id = B.building_id;
""", con)["total"].iloc[0]

# LEFT JOIN count
left_count = pd.read_sql_query("""
SELECT COUNT(*) AS total
FROM Building AS D
LEFT JOIN Benchmark AS B
    ON D.building_id = B.building_id;
""", con)["total"].iloc[0]

# Buildings without benchmarking records
buildings_without_benchmark = pd.read_sql_query("""
SELECT
    D.building_id,
    D.building_name,
    D.building_type
FROM Building AS D
LEFT JOIN Benchmark AS B
    ON D.building_id = B.building_id
WHERE B.building_id IS NULL;
""", con)

join_comparison = pd.DataFrame({
    "Join Type": ["INNER JOIN", "LEFT JOIN"],
    "Result Records": [inner_count, left_count]
})

display(join_comparison)

print(
    "Buildings without Benchmark records:",
    len(buildings_without_benchmark)
)

display(buildings_without_benchmark.head())


,Join Type,Result Records
0,INNER JOIN,38309
1,LEFT JOIN,38309


Buildings without Benchmark records: 0


,building_id,building_name,building_type


##### 6.8 Close the Database Connection

In [77]:

con.close()

print("Task 6 completed. Database connection closed.")


Task 6 completed. Database connection closed.


##### 6.9 Summary


| Join | Left rows | Right rows | Result rows | Unmatched rows | Explanation |
|---|---:|---:|---:|---:|---|
| INNER | 3,871 | 38,309 | 38,309 | 0 | Returns only matching records from both tables. All 38,309 Benchmark records have matching Building records. |
| LEFT | 3,871 | 38,309 | 38,309 | 0 | Preserves all Building records and includes matching Benchmark records. All buildings have benchmarking records, so no unmatched buildings were found. |


The join result has 38,309 rows, which is greater than the Building table's 3,871 rows because the relationship is one-to-many.

### Task 7 — SQL from Python, and SQL vs Pandas

##### 7.1 Introduction to SQL Queries from Python

In this task, two parameterized SQL queries are executed from Python. One SQL aggregation is also reproduced using Pandas to compare the results of both approaches.

##### 7.2 Query 1 – Filter Benchmark Records Using a Parameter

Question: What are the building IDs, reporting years, and Site EUI values of buildings benchmarked in a selected reporting year?

In [79]:
# Select a reporting year
import sqlite3
import pandas as pd

# Reopen the database if the Task 4 connection was closed
con = sqlite3.connect("../db/module3.db")


selected_year = 2024

query_1 = """
SELECT
    building_id,
    data_year,
    site_eui
FROM Benchmark
WHERE data_year = ?
ORDER BY site_eui DESC;
"""

result_1 = pd.read_sql_query(
    query_1,
    con,
    params=(selected_year,)
)

print("Reporting year:", selected_year)
print("Records returned:", len(result_1))

display(result_1.head(10))


Reporting year: 2024
Records returned: 3712


,building_id,data_year,site_eui
0,365,2024,969.799988
1,753,2024,756.400024
2,22479,2024,740.900024
3,49967,2024,536.099976
4,50453,2024,517.099976
5,24202,2024,494.100006
6,49709,2024,479.200012
7,49865,2024,474.700012
8,23854,2024,419.000000
9,19900,2024,409.799988


##### 7.3 Query 2 – Filter Benchmark Records Using a Parameter

Question: Which building-year records have a Site EUI greater than a selected threshold?

In [80]:

# Set the Site EUI threshold
eui_threshold = 100

query_2 = """
SELECT
    building_id,
    data_year,
    site_eui
FROM Benchmark
WHERE site_eui > ?
ORDER BY site_eui DESC;
"""

result_2 = pd.read_sql_query(
    query_2,
    con,
    params=(eui_threshold,)
)

print("Site EUI threshold:", eui_threshold)
print("Records returned:", len(result_2))

display(result_2.head(10))


Site EUI threshold: 100
Records returned: 2953


,building_id,data_year,site_eui
0,365,2023,989.200012
1,365,2024,969.799988
2,365,2018,958.500000
3,365,2022,956.700012
4,365,2019,925.099976
5,365,2017,924.400024
6,365,2020,910.900024
7,365,2021,909.900024
8,365,2016,834.400024
9,49780,2017,833.700012


##### 7.4 SQL Aggregation – Average Site EUI by Building Type

Question: What is the average Site EUI and number of Benchmark records with available Site EUI for each building type?

In [81]:

query_sql_aggregation = """
SELECT
    D.building_type,
    COUNT(B.site_eui) AS records_with_site_eui,
    ROUND(AVG(B.site_eui), 2) AS average_site_eui
FROM Building AS D
INNER JOIN Benchmark AS B
    ON D.building_id = B.building_id
GROUP BY D.building_type
ORDER BY average_site_eui DESC;
"""

sql_aggregation = pd.read_sql_query(
    query_sql_aggregation,
    con
)

display(sql_aggregation)


,building_type,records_with_site_eui,average_site_eui
0,Hospital,554,147.89
1,University,314,108.18
2,NonResidential,14313,66.70
3,Multifamily HR (10+),1377,48.77
4,School,1463,41.72
5,Multifamily MR (5-9),7531,38.06
6,Multifamily LR (1-4),11168,35.77


##### 7.5 Reproduce the SQL Aggregation Using Pandas

In [82]:

# Retrieve the database tables into DataFrames
building_data = pd.read_sql_query(
    "SELECT * FROM Building;",
    con
)

benchmark_data = pd.read_sql_query(
    "SELECT * FROM Benchmark;",
    con
)

# Join Building and Benchmark using Pandas
merged_data = pd.merge(
    building_data,
    benchmark_data,
    on="building_id",
    how="inner"
)

# Reproduce the SQL aggregation
pandas_aggregation = (
    merged_data.groupby("building_type")
    .agg(
        records_with_site_eui=("site_eui", "count"),
        average_site_eui=("site_eui", "mean")
    )
    .reset_index()
)

# Round the average Site EUI to two decimal places
pandas_aggregation["average_site_eui"] = (
    pandas_aggregation["average_site_eui"].round(2)
)

# Sort the results
pandas_aggregation = pandas_aggregation.sort_values(
    "average_site_eui",
    ascending=False
)

display(pandas_aggregation)


,building_type,records_with_site_eui,average_site_eui
0,Hospital,554,147.89
6,University,314,108.18
4,NonResidential,14313,66.70
1,Multifamily HR (10+),1377,48.77
5,School,1463,41.72
3,Multifamily MR (5-9),7531,38.06
2,Multifamily LR (1-4),11168,35.77


##### 7.6 Compare SQL and Pandas Aggregation Results

In [83]:

# Rename columns to identify their source
sql_compare = sql_aggregation.rename(columns={
    "records_with_site_eui": "sql_record_count",
    "average_site_eui": "sql_average_site_eui"
})

pandas_compare = pandas_aggregation.rename(columns={
    "records_with_site_eui": "pandas_record_count",
    "average_site_eui": "pandas_average_site_eui"
})

# Merge SQL and Pandas results
comparison = pd.merge(
    sql_compare,
    pandas_compare,
    on="building_type",
    how="outer"
)

# Calculate differences
comparison["record_count_difference"] = (
    comparison["sql_record_count"]
    - comparison["pandas_record_count"]
)

comparison["average_eui_difference"] = (
    comparison["sql_average_site_eui"]
    - comparison["pandas_average_site_eui"]
)

display(comparison)


,building_type,sql_record_count,sql_average_site_eui,pandas_record_count,pandas_average_site_eui,record_count_difference,average_eui_difference
0,Hospital,554,147.89,554,147.89,0,0.0
1,Multifamily HR (10+),1377,48.77,1377,48.77,0,0.0
2,Multifamily LR (1-4),11168,35.77,11168,35.77,0,0.0
3,Multifamily MR (5-9),7531,38.06,7531,38.06,0,0.0
4,NonResidential,14313,66.70,14313,66.70,0,0.0
5,School,1463,41.72,1463,41.72,0,0.0
6,University,314,108.18,314,108.18,0,0.0


##### 7.7 Findings and Summary

Two parameterized SQL queries were successfully executed from Python. The first query returned 3,712 Benchmark records for 2024, while the second returned 2,953 records with Site EUI greater than 100.

The SQL aggregation and Pandas aggregation produced identical results. Both methods returned the same record counts and average Site EUI for all seven building types, with zero differences in the comparison table.

Hospitals had the highest average Site EUI at 147.89 kBtu/sf, followed by Universities at 108.18 kBtu/sf. Multifamily LR (1–4) had the lowest average at 35.77 kBtu/sf.

Overall, this task demonstrates that SQL and Pandas can produce consistent aggregation results when the same join, grouping, and missing-value handling are applied.

### Interpretation

1. Does each proposed key identify a row uniquely? Show the evidence.
 The proposed primary keys identify records uniquely in their respective tables. The Building table uses building_id as its primary key, while the Benchmark table uses the combination of building_id and data_year as its composite primary key.
 Before loading, the key validation showed zero duplicate keys and zero missing primary key values. This supports the uniqueness of the proposed keys.

2. What does one row represent in each table, and are the grains compatible for the join?
 One row in the Building table represents one unique building, resulting in 3,871 records. One row in the Benchmark table represents one building's energy benchmarking record for a particular reporting year, resulting in 38,309 records.
 The grains are compatible because one building can have multiple annual benchmarking records. The one-to-many relationship allows the building information to be combined with its corresponding yearly energy measurements using building_id.

3. How could duplicate keys or a many-to-many relationship inflate aggregated values?
 Duplicate keys or an incorrect many-to-many relationship can multiply records during a join. When multiple rows match on both sides, the joined result may contain repeated combinations, which can inflate counts, sums, and other aggregated values.
 In this project, the unique Building primary key and the composite Benchmark primary key help prevent this problem. The one-to-many relationship ensures that each Benchmark record matches only one Building record.

4. What information is lost with an INNER JOIN compared with a LEFT JOIN?
An INNER JOIN keeps only records with matching keys in both tables. A LEFT JOIN preserves all records from the left table, including those without matching records on the right.
In this project, both joins returned 38,309 rows because all Benchmark records matched a Building record and all 3,871 buildings had benchmarking records. Therefore, no unmatched building or benchmarking records were lost in the tested join.

5. When was SQL more suitable than Pandas for your task, and when was it less suitable?
 SQL was more suitable for creating tables, defining primary and foreign keys, joining relational tables, filtering records, and performing aggregations directly in the database. It allowed the queries to be executed without loading the entire dataset into Python.
 Pandas was more suitable for data preparation, checking attribute consistency, inspecting records, and comparing aggregation results. It also provided a convenient way to display and analyze query results using DataFrames.
 Both SQL and Pandas produced identical aggregation results in Task 7, showing that they can be used together effectively.

### Schema, ER Sketch and its key explanation

##### DATABASE SCHEMA

 `Building` Table

| Column | Data type | Key |
| :--- | :--- | :---: |
| `building_id` | INTEGER | PK |
| `building_name` | TEXT | |
| `building_type` | TEXT | |
| `address` | TEXT | |
| `property_gfa_total` | REAL | |

 `Benchmark` Table

| Column | Data type | Key |
| :--- | :--- | :---: |
| `building_id` | INTEGER | PK, FK |
| `data_year` | INTEGER | PK |
| `site_eui` | REAL | |
| `source_eui` | REAL | |
| `site_energy_use` | REAL | |
| `total_ghg_emissions` | REAL | |


#### ER Sketch

```mermaid
erDiagram
    BUILDING ||--o{ BENCHMARK : "has yearly records"

    BUILDING {
        INTEGER building_id PK
        TEXT building_name
        TEXT building_type
        TEXT address
        REAL property_gfa_total
    }

    BENCHMARK {
        INTEGER building_id PK, FK
        INTEGER data_year PK
        REAL site_eui
        REAL source_eui
        REAL site_energy_use
        REAL total_ghg_emissions
    }
```


```text
+--------------------------+          +-----------------------------+

|         BUILDING         |          |          BENCHMARK          |
+--------------------------+          +-----------------------------+

| PK  building_id (INT)    |----1:N---| PK, FK building_id (INT)    |
|     building_name (TEXT) |          | PK     data_year (INT)      |
|     building_type (TEXT) |          |        site_eui (REAL)      |
|     address (TEXT)       |          |        source_eui (REAL)    |
|     property_gfa_total   |          |        site_energy_use      |
+--------------------------+          |        total_ghg_emissions  |
                                      +-----------------------------+
```


Key Explanation



The database consists of two relational tables: Building and Benchmark.

The Building table uses `building_id` as its primary key because each building is represented by one unique record. The Benchmark table uses a composite primary key consisting of `building_id` and `data_year`, because a building can have multiple benchmarking records across different reporting years.

The `building_id` column in Benchmark is also a foreign key referencing `Building.building_id`. This establishes a one-to-many relationship in which one building can have multiple annual benchmarking records.

The schema separates building characteristics from annual energy measurements, reducing unnecessary repetition of building information and supporting efficient relational queries.